# Field Validation: Undercharge (Site 2) and Compressor Staging (Site 1)

## Purpose

Per the original project scope, the Field dataset is reserved exclusively for
validation once a model exists - never as training data (only 1-2 real fault
instances per site, statistically meaningless as training data on their own).

This notebook checks whether ANYTHING learned from the Simulated-dataset undercharge
investigation transfers to this real 40% undercharge case - a genuinely important
test given undercharge was the one Simulated-dataset fault whose model was found
NOT production-usable (unresolved forward-in-time collapse).

## Real, honest expectations going in

- This is a single real case (n=1 fault instance), so this cannot be a statistically
  rigorous test - it's a sanity check, not a validation in the formal sense.
- Column names and sensor set will very likely differ from both prior datasets (real
  field-installed sensors, per the LBNL documentation's field-sensor table) - check
  before assuming anything carries over.
- Given undercharge's known issues, a negative result here would not be surprising -
  document honestly either way, don't force a positive spin.

In [1]:
import pandas as pd

site2_fault = pd.read_csv("../data/raw/field/Site2_Undercharged40.csv")
site2_baseline = pd.read_csv("../data/raw/field/Site2_Unfaulted.csv")

print("=== Site2_Undercharged40.csv ===")
print(f"Shape: {site2_fault.shape}")
print(f"Columns: {list(site2_fault.columns)}")
print("\nFirst few rows:")
print(site2_fault.head())

print("\n=== Site2_Unfaulted.csv ===")
print(f"Shape: {site2_baseline.shape}")

=== Site2_Undercharged40.csv ===
Shape: (41522, 26)
Columns: ['Datetime', 'RTU_COMP_WATT', 'RTU_LA_COND_TEMP', 'RTU_MA_HUM', 'RTU_MA_TEMP', 'RTU_OA_HUM', 'RTU_OA_TEMP', 'RTU_RA_HUM', 'RTU_RA_TEMP', 'RTU_REFG_COND_TEMP_1', 'RTU_REFG_COND_TEMP_2', 'RTU_REFG_DISC_PRES_1', 'RTU_REFG_DISC_PRES_2', 'RTU_REFG_DISC_TEMP_1', 'RTU_REFG_DISC_TEMP_2', 'RTU_REFG_SUCT_PRES_1', 'RTU_REFG_SUCT_PRES_2', 'RTU_REFG_SUCT_TEMP_1', 'RTU_REFG_SUCT_TEMP_2', 'RTU_SA_FAN_WATT', 'RTU_SA_FLOW', 'RTU_SA_HUM', 'RTU_SA_TEMP', 'RTU_TOT_WATT', 'ZA_HUM', 'ZA_TEMP']

First few rows:
              Datetime  RTU_COMP_WATT  RTU_LA_COND_TEMP  RTU_MA_HUM  \
0  2020-06-01 00:00:00            0.0          7.347733   59.704639   
1  2020-06-01 00:01:00            0.0          7.484430   59.859142   
2  2020-06-01 00:02:00            0.0          7.435305   60.045574   
3  2020-06-01 00:03:00            0.0          7.275113   60.311497   
4  2020-06-01 00:04:00            0.0          7.317831   60.643982   

   RTU_MA_TEMP  RT

## Comparing baseline vs. fault means, both circuits, to identify which circuit is
## actually faulted (not labeled directly in the file)

In [2]:
compare_cols = [
    "RTU_REFG_SUCT_PRES_1", "RTU_REFG_SUCT_TEMP_1", "RTU_REFG_DISC_PRES_1", "RTU_REFG_DISC_TEMP_1",
    "RTU_REFG_SUCT_PRES_2", "RTU_REFG_SUCT_TEMP_2", "RTU_REFG_DISC_PRES_2", "RTU_REFG_DISC_TEMP_2",
    "RTU_COMP_WATT", "RTU_TOT_WATT",
]

baseline_means = site2_baseline[compare_cols].mean()
fault_means = site2_fault[compare_cols].mean()

comparison = pd.DataFrame({"baseline": baseline_means, "fault": fault_means})
comparison["pct_change"] = (comparison["fault"] / comparison["baseline"] - 1) * 100
print(comparison)

                         baseline        fault  pct_change
RTU_REFG_SUCT_PRES_1     6.664155     7.066308    6.034575
RTU_REFG_SUCT_TEMP_1    19.255222    19.175938   -0.411750
RTU_REFG_DISC_PRES_1    10.451605     9.890542   -5.368198
RTU_REFG_DISC_TEMP_1    42.084470    35.709744  -15.147455
RTU_REFG_SUCT_PRES_2     7.118805     6.762715   -5.002105
RTU_REFG_SUCT_TEMP_2    21.807387    22.195069    1.777759
RTU_REFG_DISC_PRES_2     8.424844     8.003109   -5.005844
RTU_REFG_DISC_TEMP_2    28.912991    31.994213   10.656879
RTU_COMP_WATT         1590.001184  1131.074900  -28.863267
RTU_TOT_WATT          3087.259279  2390.396025  -22.572230


## Critical caution before interpreting anything: fault and baseline periods are
## different real months/years — the same weather confound theme as the whole
## Simulated-dataset investigation, now in real field data

Per LBNL's documentation: Site2_Undercharged40.csv = June 1-29, 2020.
Site2_Unfaulted.csv = July 31-Sept 30, 2020 AND June-Sept 2021. These are
DIFFERENT real time periods - any comparison of raw means conflates the fault's
real effect with real seasonal/weather differences between June and July-
September, and even between 2020 and 2021. This is not a hypothetical concern -
it is the single most consistent lesson of this entire project. Treating any
raw difference as "the fault's effect" without checking outdoor conditions
first would repeat a mistake already made and corrected multiple times earlier.

**Circuit assignment is also genuinely ambiguous, not confidently determined**:
- Circuit 1: suction pressure +6.03% (matches undercharge's established direction),
  but suction temp -0.41% (does NOT match the expected up-direction), discharge
  temp -15.15%.
- Circuit 2: suction pressure -5.00% (does NOT match), suction temp +1.78%
  (weakly matches), discharge temp +10.66%.

Neither circuit shows a clean, fully consistent undercharge signature - real,
honest ambiguity, not a confident circuit identification. This should be stated
plainly rather than picking whichever circuit "looks more like" undercharge and
presenting that choice as settled.

**RTU_COMP_WATT/RTU_TOT_WATT's large drop (-28.9%/-22.6%) is the most dramatic
number here, but is exactly the kind of result most vulnerable to the period-
mismatch confound** - lower power draw could mean reduced compressor effort due
to undercharge, or could simply mean June was cooler/less humid than the
July-September comparison window. Cannot be trusted without checking outdoor
conditions directly.

In [3]:
print("Site2_Unfaulted RTU_OA_TEMP by month/year:")
site2_baseline["Datetime"] = pd.to_datetime(site2_baseline["Datetime"])
print(site2_baseline.groupby(site2_baseline["Datetime"].dt.to_period("M"))["RTU_OA_TEMP"].mean())

print("\nSite2_Undercharged40 RTU_OA_TEMP (June 2020):")
site2_fault["Datetime"] = pd.to_datetime(site2_fault["Datetime"])
print(site2_fault["RTU_OA_TEMP"].mean())

Site2_Unfaulted RTU_OA_TEMP by month/year:
Datetime
2020-07    23.914509
2020-08    18.944078
2020-09    13.076121
2021-06    19.010479
2021-07    17.588025
2021-08    17.723818
2021-09    13.338473
Freq: M, Name: RTU_OA_TEMP, dtype: float64

Site2_Undercharged40 RTU_OA_TEMP (June 2020):
19.337583742690622


## Weather check: fault period's outdoor temperature is not obviously extreme
## relative to baseline months — partial reassurance, not full resolution

Fault period (June 2020) mean OA_TEMP: 19.34°C. Baseline months range 13.1-23.9°C,
with 2021-06 (19.01°C) and 2021-08 (17.72°C) both close to the fault period's value.
This is a genuine, partial reassurance - the confound isn't as blunt as "comparing
a heat wave to a cold snap." But this only checks the MEAN outdoor temperature per
month, not the full distribution or other confounding factors (humidity, day-to-day
variability, occupancy patterns, real equipment aging/drift between 2020 and 2021)
- a real limitation of this check, not a full resolution of the period-mismatch
concern.

**Most defensible comparison available**: baseline should be restricted to the
closest-matching real period(s) - 2021-06 and/or 2021-08 - rather than pooling
across all baseline months indiscriminately, since some baseline months (2020-09,
2021-09 at ~13°C) are clearly much cooler than the fault period and would
introduce their own real confound if included.

In [4]:
site2_baseline_matched = site2_baseline[site2_baseline["Datetime"].dt.to_period("M") == "2021-06"]

print(f"Matched baseline (2021-06) shape: {site2_baseline_matched.shape}")
print(f"Matched baseline mean OA_TEMP: {site2_baseline_matched['RTU_OA_TEMP'].mean():.2f}")

comparison_matched = pd.DataFrame({
    "baseline_2021_06": site2_baseline_matched[compare_cols].mean(),
    "fault_2020_06": site2_fault[compare_cols].mean(),
})
comparison_matched["pct_change"] = (comparison_matched["fault_2020_06"] / comparison_matched["baseline_2021_06"] - 1) * 100
print(comparison_matched)

Matched baseline (2021-06) shape: (43085, 26)
Matched baseline mean OA_TEMP: 19.01
                      baseline_2021_06  fault_2020_06  pct_change
RTU_REFG_SUCT_PRES_1          6.720213       7.066308    5.150058
RTU_REFG_SUCT_TEMP_1         19.210153      19.175938   -0.178104
RTU_REFG_DISC_PRES_1         10.675755       9.890542   -7.355101
RTU_REFG_DISC_TEMP_1         42.036282      35.709744  -15.050184
RTU_REFG_SUCT_PRES_2          6.993148       6.762715   -3.295132
RTU_REFG_SUCT_TEMP_2         22.916821      22.195069   -3.149441
RTU_REFG_DISC_PRES_2          8.510747       8.003109   -5.964664
RTU_REFG_DISC_TEMP_2         31.252789      31.994213    2.372347
RTU_COMP_WATT              1618.763647    1131.074900  -30.127236
RTU_TOT_WATT               3130.599376    2390.396025  -23.644142


## Real, encouraging finding: Circuit 1 matches the established undercharge
## signature reasonably well, even after weather-matching — Circuit 2 does not

| Signal | Circuit 1 | Matches undercharge signature? | Circuit 2 | Matches? |
|---|---|---|---|
| Suction pressure | +5.15% | Yes (expected: up) | -3.30% | No |
| Suction temp | -0.18% | Ambiguous (expected: up, but near-zero) | -3.15% | No |
| Discharge pressure | -7.36% | Yes (expected: down) | -5.96% | Yes (weaker signal) |

**Circuit 1 is very likely the real faulted circuit (Circuit B)** - 2 of 3 signals
match the direction established in notebook 01's Simulated-dataset EDA, even after
restricting the comparison to weather-matched months. This is a genuinely
meaningful, positive result: the qualitative physical signature of undercharge
(suction pressure rising, discharge pressure falling) appears to hold in real field
data too, not just in simulation - even though the SIMULATED-DATASET MODEL trained
on this fault was found not production-usable due to an unrelated generalization
problem (time-based instability). The underlying physics-based EDA finding and the
specific trained classifier are two separate things, and this checks the former,
not the latter.

**Power draw dropped substantially even after weather-matching** (RTU_COMP_WATT
-30.1%, RTU_TOT_WATT -23.6%) - plausible and consistent with reduced compressor
effort under a genuinely lower refrigerant charge, though month-level weather
matching doesn't rule out finer-grained (day-to-day) confounds - a real limitation
of this check, worth stating rather than treating this as fully controlled.

**Important, honest scope limitation**: this is a single real case (n=1), with
inherent ambiguity in circuit identification (not confirmed via documentation,
inferred from signal direction), and weather-matched only at the monthly level.
This is a real, meaningful, encouraging sanity check - not a rigorous statistical
validation, and should not be oversold as "the model works on real data." What it
DOES support: the underlying physical reasoning behind undercharge's EDA findings
transfers to a real system, which is valuable and worth documenting honestly at
that scope.

## Bounded check: is the power-draw drop a sustained daily pattern, or driven by a
## few unusual days?

Per this project's established practice (e.g. condenser fouling's capacity wobble,
notebook 03) - a mean can hide whether an effect is real and sustained or an
artifact of a few outlier days. Checking the daily pattern once, directly, before
finalizing this validation's conclusion.

In [5]:
site2_fault_daily = site2_fault.set_index("Datetime")["RTU_TOT_WATT"].resample("D").mean()
site2_baseline_matched_daily = site2_baseline_matched.set_index("Datetime")["RTU_TOT_WATT"].resample("D").mean()

print("Fault period (June 2020) daily RTU_TOT_WATT:")
print(site2_fault_daily.describe())
print("\nMatched baseline (June 2021) daily RTU_TOT_WATT:")
print(site2_baseline_matched_daily.describe())

Fault period (June 2020) daily RTU_TOT_WATT:
count      30.000000
mean     2319.526315
std      1541.312860
min        40.000004
25%      1363.568668
50%      2010.534594
75%      3329.145798
max      5335.199861
Name: RTU_TOT_WATT, dtype: float64

Matched baseline (June 2021) daily RTU_TOT_WATT:
count      30.000000
mean     3130.838648
std      1969.011516
min       403.231922
25%      1935.032718
50%      2506.153718
75%      3976.279985
max      7376.965951
Name: RTU_TOT_WATT, dtype: float64


## Confirmed: sustained pattern across the whole distribution, not a few outlier days

| | Fault (June 2020) | Baseline (June 2021) |
|---|---|---|
| Min | 40.0 | 403.2 |
| 25th percentile | 1363.6 | 1935.0 |
| Median | 2010.5 | 2506.2 |
| 75th percentile | 3329.1 | 3976.3 |
| Max | 5335.2 | 7377.0 |

Every quantile of the fault period sits meaningfully below the corresponding
quantile of the matched baseline period - a real, sustained shift across the whole
distribution, not driven by a handful of unusually low days. This adds real
confidence to the power-draw finding: it's a consistent, day-to-day pattern of
reduced power consumption throughout the fault period, consistent with reduced
compressor effort under genuine undercharge, not an artifact.

## Summary: Field validation, undercharge (Site 2, Colchester CT)

**Purpose**: sanity-check whether the Simulated-dataset's undercharge EDA findings
transfer to a real field case (n=1, per the original project scope reserving Field
data exclusively for validation).

**Real, important methodological step**: recognized and corrected for a genuine
period-mismatch confound (fault period: June 2020; baseline: July-Sept 2020/2021) -
the same weather-confound discipline established throughout this entire project's
Simulated-dataset work, now correctly applied to real field data too.

**Findings, after weather-matching to the closest comparable month (June 2021)**:
- Circuit 1 (inferred as the real faulted circuit, not directly labeled in the
  data) shows suction pressure UP (+5.15%) and discharge pressure DOWN (-7.36%) -
  matching the direction established in notebook 01's Simulated-dataset EDA.
  Suction temperature was ambiguous (near-zero change).
- Circuit 2 does not match the expected direction on 2 of 3 signals - reinforces
  that Circuit 1 is the more likely faulted circuit.
- Power draw (RTU_COMP_WATT, RTU_TOT_WATT) dropped substantially and consistently
  across the ENTIRE distribution (confirmed via daily resampling, not just the
  mean) - a real, sustained effect, not a few outlier days.

**Honest scope**: n=1 real case, circuit identity inferred rather than confirmed,
weather-matched only at the monthly level. This is a genuine, encouraging sanity
check on the underlying PHYSICAL REASONING behind undercharge's EDA findings - it
is NOT a validation of the trained Simulated-dataset classifier itself, which
remains not production-usable due to its separate, unresolved forward-in-time
generalization problem. These are two different questions: "does the physics
transfer to reality" (this notebook: encouragingly, yes) vs. "does the specific
trained model work reliably" (still no, per FINAL_MODEL_METRICS.md).

## Field Validation: Compressor Staging Fault (Site 1, Milford CT)

## Fault description, per LBNL documentation

"Compressor Controls Fault, Stage 2 will not engage" - Site 1 (Milford, CT,
restaurant). Fault period: June 9 - Aug 8, 2021. Unfaulted comparison: Aug 10 -
Sep 29, 2021. Unlike undercharge, this is a controls/logic fault (a stage simply
failing to activate when it should), conceptually closer to the Experimental
dataset's OA damper stuck (a component stuck rather than a charge-level physical
problem) than to anything in the Simulated dataset - no direct Simulated-dataset
model to compare against here, this is purely a fresh, honest look at real data.

## Same period-mismatch discipline applies

Fault period: June-Aug 2021. Baseline: Aug-Sep 2021. These are ADJACENT but not
identical months - less of a mismatch than Site 2's case, but still worth checking
outdoor conditions before trusting any raw comparison, same discipline as before.

In [6]:
site1_fault = pd.read_csv("../data/raw/field/Site1_Staging_Fault.csv")
site1_baseline = pd.read_csv("../data/raw/field/Site1_Unfaulted.csv")

print("=== Site1_Staging_Fault.csv ===")
print(f"Shape: {site1_fault.shape}")
print(f"Columns: {list(site1_fault.columns)}")

print("\n=== Site1_Unfaulted.csv ===")
print(f"Shape: {site1_baseline.shape}")

site1_fault["Datetime"] = pd.to_datetime(site1_fault["Datetime"])
site1_baseline["Datetime"] = pd.to_datetime(site1_baseline["Datetime"])

print("\nFault period OA_TEMP by month:")
print(site1_fault.groupby(site1_fault["Datetime"].dt.to_period("M"))["RTU_OA_TEMP"].mean())
print("\nBaseline period OA_TEMP by month:")
print(site1_baseline.groupby(site1_baseline["Datetime"].dt.to_period("M"))["RTU_OA_TEMP"].mean())

=== Site1_Staging_Fault.csv ===
Shape: (86354, 26)
Columns: ['Datetime', 'RTU_COMP_WATT', 'RTU_LA_COND_TEMP', 'RTU_MA_HUM', 'RTU_MA_TEMP', 'RTU_OA_HUM', 'RTU_OA_TEMP', 'RTU_RA_HUM', 'RTU_RA_TEMP', 'RTU_REFG_COND_TEMP_1', 'RTU_REFG_COND_TEMP_2', 'RTU_REFG_DISC_PRES_1', 'RTU_REFG_DISC_PRES_2', 'RTU_REFG_DISC_TEMP_1', 'RTU_REFG_DISC_TEMP_2', 'RTU_REFG_SUCT_PRES_1', 'RTU_REFG_SUCT_PRES_2', 'RTU_REFG_SUCT_TEMP_1', 'RTU_REFG_SUCT_TEMP_2', 'RTU_SA_FAN_WATT', 'RTU_SA_FLOW', 'RTU_SA_HUM', 'RTU_SA_TEMP', 'RTU_TOT_WATT', 'ZA_HUM', 'ZA_TEMP']

=== Site1_Unfaulted.csv ===
Shape: (73099, 26)

Fault period OA_TEMP by month:
Datetime
2021-06    23.855889
2021-07    24.642145
2021-08    23.600072
Freq: M, Name: RTU_OA_TEMP, dtype: float64

Baseline period OA_TEMP by month:
Datetime
2021-08    26.342842
2021-09    21.313497
Freq: M, Name: RTU_OA_TEMP, dtype: float64


## Weather check: much closer overlap than Site 2's case — restricting to the
## shared month (August 2021) in both periods for the fairest comparison

In [7]:
site1_fault_aug = site1_fault[site1_fault["Datetime"].dt.to_period("M") == "2021-08"]
site1_baseline_aug = site1_baseline[site1_baseline["Datetime"].dt.to_period("M") == "2021-08"]

print(f"Fault August subset: {site1_fault_aug.shape}, mean OA_TEMP: {site1_fault_aug['RTU_OA_TEMP'].mean():.2f}")
print(f"Baseline August subset: {site1_baseline_aug.shape}, mean OA_TEMP: {site1_baseline_aug['RTU_OA_TEMP'].mean():.2f}")

compare_cols_site1 = [
    "RTU_REFG_SUCT_PRES_1", "RTU_REFG_SUCT_TEMP_1", "RTU_REFG_DISC_PRES_1", "RTU_REFG_DISC_TEMP_1",
    "RTU_REFG_SUCT_PRES_2", "RTU_REFG_SUCT_TEMP_2", "RTU_REFG_DISC_PRES_2", "RTU_REFG_DISC_TEMP_2",
    "RTU_COMP_WATT", "RTU_TOT_WATT",
]

comparison_site1 = pd.DataFrame({
    "baseline_aug": site1_baseline_aug[compare_cols_site1].mean(),
    "fault_aug": site1_fault_aug[compare_cols_site1].mean(),
})
comparison_site1["pct_change"] = (comparison_site1["fault_aug"] / comparison_site1["baseline_aug"] - 1) * 100
print(comparison_site1)

Fault August subset: (10185, 26), mean OA_TEMP: 23.60
Baseline August subset: (31541, 26), mean OA_TEMP: 26.34
                      baseline_aug    fault_aug  pct_change
RTU_REFG_SUCT_PRES_1      6.551719     6.475893   -1.157346
RTU_REFG_SUCT_TEMP_1     22.783209    22.201726   -2.552242
RTU_REFG_DISC_PRES_1     12.472565    10.939113  -12.294593
RTU_REFG_DISC_TEMP_1     56.029349    48.885035  -12.751022
RTU_REFG_SUCT_PRES_2      8.053194     9.168227   13.845844
RTU_REFG_SUCT_TEMP_2     24.508880    23.349224   -4.731577
RTU_REFG_DISC_PRES_2     11.840037     9.240728  -21.953559
RTU_REFG_DISC_TEMP_2     44.477038    24.802744  -44.234722
RTU_COMP_WATT          1730.059642  1346.441493  -22.173695
RTU_TOT_WATT           3921.903417  2357.264097  -39.894897


## Residual weather mismatch even within the "same month": a real, honest caveat

Restricting to August in both periods narrowed but did not eliminate the weather
gap - fault period mean OA_TEMP 23.60°C vs. baseline 26.34°C, a real ~2.7°C
difference. Notably, baseline is the WARMER period here - meaning some of the
observed reduced power draw in the fault period could reflect genuinely lower
cooling demand, not solely the staging fault. This doesn't invalidate the finding,
but it means the magnitude of the effect (not just its existence) should be
interpreted cautiously.

## Findings: Circuit 2 shows a dramatic effect consistent with 'Stage 2 will not
## engage' — Circuit 1 shows a much more modest effect

| Signal | Circuit 1 | Circuit 2 |
|---|---|---|
| Suction pressure | -1.16% | +13.85% |
| Suction temp | -2.55% | -4.73% |
| Discharge pressure | -12.29% | -21.95% |
| Discharge temp | -12.75% | **-44.23%** |

**Circuit 2's changes are far larger in magnitude than Circuit 1's across every
signal** - especially discharge temperature (-44.2%) and discharge pressure
(-22.0%). This is exactly the pattern a real "stage 2 will not engage" fault should
produce: if the second compressor stage genuinely isn't activating when it should,
its own circuit's discharge conditions (driven directly by that compressor
running) would show a dramatic reduction, while the still-functioning Circuit 1
(stage 1) shows a comparatively modest, more weather-driven change. This makes
Circuit 2 the very likely "Stage 2" circuit - a plausible, physically sensible
inference, though (same honesty standard as Site 2) not directly confirmed by
the file's documentation.

**RTU_TOT_WATT dropped substantially (-39.9%)** - a real, large effect, though its
exact magnitude is muddied by the residual ~2.7°C weather gap noted above (baseline
being the warmer period would itself reduce fault-period power draw somewhat,
independent of the fault).

## Summary: Field validation, compressor staging fault (Site 1, Milford CT)

**No direct Simulated-dataset model exists for this fault type** - this was a
fresh, exploratory look at real data, not a check against an established EDA
signature (unlike the undercharge case).

**Real, physically plausible finding**: Circuit 2 shows dramatically larger
reductions than Circuit 1 across every refrigerant signal (especially discharge
temp, -44.2%), consistent with a "stage 2 will not engage" fault specifically
affecting that circuit's operation, while Circuit 1 (inferred as stage 1) shows
comparatively modest changes.

**Honest caveat**: even after restricting to the closest-matching month, a
residual ~2.7°C weather difference remains (baseline was the warmer period),
meaning the exact MAGNITUDE of the power-draw reduction (-39.9%) should be
interpreted cautiously - some of it likely reflects genuinely lower cooling
demand, not solely the fault. The qualitative pattern (Circuit 2 far more
affected than Circuit 1) is the more defensible finding here than the precise
percentage figures.

**Scope**: n=1 real case, no formal statistical test, circuit identity inferred
rather than confirmed. A genuine, informative first look at this fault type in
real data - not a validation of any specific trained model, since none exists
for this fault type in this project.